# LF5 — vision_model: bỏ phiếu độ hữu dụng bằng Claude vision

Đây là **một labeling function (LF5)** trong khung weak-supervision (xem `paper/Labeling_Plan.md`),
độ tin **trung bình** — KHÔNG phải nhãn cuối. Nhãn cuối do label model hợp nhất LF1–LF8; tín hiệu
mạnh nhất là **correctness vs GT** (LF1–3, `scripts/label_correctness.py`).

## 3 tác vụ (xem paper §3.2)
- `1_maturity_evaluation` — độ chín (thấy rõ trái, màu/hình/kích thước vỏ trung thực).
- `2_foliar_disease` — bệnh lá (Gray Leaf Spot, Leaf Rot): thấy rõ phiến lá + triệu chứng.
- `3_trunk_crown_disease` — bệnh thân/ngọn (Stem Bleeding, Bud Rot, Bud Root Dropping):
  thấy rõ thân/gốc/ngọn + dấu hiệu (chảy nhựa thân, thối/rụng đọt).

## LF5 bỏ phiếu mỗi tác vụ: `1` / `0` / `abstain`
`conf ≥ τ_high → 1` (hữu dụng), `conf ≤ τ_low → 0` (không), giữa → `abstain` (để LF khác/label model quyết).

Tầng 1 (cổng chất lượng tất định) chạy trước — ảnh rớt cổng: LF5 abstain toàn bộ (thuộc miền chất lượng).

## 1. API key

In [ ]:
# %pip install anthropic pillow pandas numpy
import os
assert os.environ.get('ANTHROPIC_API_KEY'), 'Chưa set ANTHROPIC_API_KEY'

## 2. Cấu hình

In [ ]:
import base64, io
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image, ImageOps
import anthropic

ROOT = Path.cwd()
if not (ROOT/'Dataset').exists():
    for p in ROOT.parents:
        if (p/'Dataset').exists(): ROOT=p; break
OUT = ROOT/'labels'; OUT.mkdir(exist_ok=True)

MODEL='claude-opus-4-8'; MAX_LONG_EDGE=1024; N_PER_SOURCE=16
TAU_HIGH, TAU_LOW = 0.70, 0.30      # ngưỡng vote của LF5 (1 / abstain / 0)

# cổng chất lượng (tất định)
GATE_PROC_DIM=512; GATE_MIN_DIM=200; GATE_BLUR_MIN=60.0
GATE_DARK_MAX=0.60; GATE_BRIGHT_MAX=0.45; GATE_MEAN_MIN=25; GATE_MEAN_MAX=235

# 3 tác vụ, đúng thứ tự AGENTS.md
TASKS=['1_maturity_evaluation','2_foliar_disease','3_trunk_crown_disease']  # nhãn chuẩn (khớp label_correctness.ipynb)
# pydantic không cho định danh bắt đầu bằng số -> ánh xạ nhãn chuẩn sang tên trường của mô hình:
FIELD={'1_maturity_evaluation':'maturity_evaluation','2_foliar_disease':'foliar_disease','3_trunk_crown_disease':'trunk_crown_disease'}
IMG_EXTS={'.jpg','.jpeg','.png','.bmp','.webp'}
# ánh xạ thư mục -> tác vụ native (LF6, để đối chiếu)
FOLDER_NATIVE={'Gray Leaf Spot':{'2_foliar_disease'},'Leaf Rot':{'2_foliar_disease'},
  'Stem Bleeding':{'3_trunk_crown_disease'},'Bud Rot':{'3_trunk_crown_disease'},
  'Bud Root Dropping':{'3_trunk_crown_disease'},'coconut-veirf-v5':{'1_maturity_evaluation'}}
client=anthropic.Anthropic()

## 3. Tầng 1 — cổng chất lượng tất định (miễn phí)

In [ ]:
_LAP=np.array([[0,1,0],[1,-4,1],[0,1,0]],dtype=np.float32)
def _lapvar(g):
    p=np.pad(g,1,mode='reflect'); out=np.zeros_like(g)
    for dy in range(3):
        for dx in range(3): out+=_LAP[dy,dx]*p[dy:dy+g.shape[0],dx:dx+g.shape[1]]
    return float(out.var())
def quality_gate(path):
    img=ImageOps.exif_transpose(Image.open(path)).convert('RGB'); w,h=img.size
    s=GATE_PROC_DIM/max(w,h) if max(w,h)>GATE_PROC_DIM else 1.0
    small=img.resize((max(1,int(w*s)),max(1,int(h*s)))) if s<1 else img
    arr=np.asarray(small,dtype=np.float32); g=arr@np.array([.299,.587,.114],dtype=np.float32)
    lap,mean=_lapvar(g),float(g.mean()); dark,bright=float((g<15).mean()),float((g>245).mean())
    r=[]
    if min(w,h)<GATE_MIN_DIM: r.append('lowres')
    if lap<GATE_BLUR_MIN: r.append('blur')
    if dark>GATE_DARK_MAX or mean<GATE_MEAN_MIN: r.append('underexposed')
    if bright>GATE_BRIGHT_MAX or mean>GATE_MEAN_MAX: r.append('overexposed')
    return {'quality_pass':not r,'gate_reason':'|'.join(r),'lap_var':round(lap,1),'mean_bright':round(mean,1),'min_dim':min(w,h)}

## 4. Rubric + schema (3 tác vụ)

In [ ]:
SYSTEM='''Bạn là giám định viên ảnh nông nghiệp. Với ảnh cây/trái dừa (đã qua bộ lọc chất lượng cơ bản),
đánh giá ĐỘC LẬP ảnh có đủ thông tin thị giác để mỗi tác vụ sau THÀNH CÔNG hay không. Chỉ dựa nội dung quan sát.
Đặt suitable=true CHỈ KHI ảnh thực sự dùng được cho tác vụ đó:
- maturity_evaluation: thấy rõ >=1 trái dừa, kích thước/hình dạng/màu vỏ được tái hiện trung thực để phán đoán độ chín (dry/green/tender).
- foliar_disease: thấy rõ phiến lá/tàu lá và triệu chứng bệnh trên lá (đốm, cháy, thối lá) đủ nét.
- trunk_crown_disease: thấy rõ thân/gốc hoặc ngọn/đọt và dấu hiệu bệnh thân-ngọn (chảy nhựa thân, thối/rụng đọt) đủ nét.
Nếu đối tượng cần thiết không hiện diện hoặc bị che khuất/kém quá mức cho một tác vụ -> suitable=false.
confidence là số thực 0..1 = khả năng ảnh phù hợp tác vụ đó.'''
from pydantic import BaseModel
class Task(BaseModel):
    suitable: bool; confidence: float; reason: str
class ImageLabel(BaseModel):
    maturity_evaluation: Task; foliar_disease: Task; trunk_crown_disease: Task

## 5. Lấy mẫu

In [ ]:
def collect():
    d0=ROOT/'Dataset'/'Coconut Tree Disease Dataset'
    for f in ['Gray Leaf Spot','Leaf Rot','Stem Bleeding','Bud Rot','Bud Root Dropping']:
        d=d0/f
        if d.is_dir(): yield f, sorted(p for p in d.iterdir() if p.suffix.lower() in IMG_EXTS)
    imgs=[]
    for sp in ['train','valid','test']:
        d=ROOT/'Dataset'/'coconut-veirf-v5'/sp/'images'
        if d.is_dir(): imgs+=sorted(p for p in d.iterdir() if p.suffix.lower() in IMG_EXTS)
    yield 'coconut-veirf-v5', imgs
def sample_even(a,n):
    if len(a)<=n: return a
    st=len(a)/n; return [a[int(i*st)] for i in range(n)]
pilot=[(p,f) for f,imgs in collect() for p in sample_even(imgs,N_PER_SOURCE)]
print(len(pilot),'ảnh'); pd.Series([f for _,f in pilot]).value_counts()

## 6. Gọi model

In [ ]:
def encode(path):
    img=ImageOps.exif_transpose(Image.open(path)).convert('RGB'); w,h=img.size
    if max(w,h)>MAX_LONG_EDGE: s=MAX_LONG_EDGE/max(w,h); img=img.resize((int(w*s),int(h*s)))
    b=io.BytesIO(); img.save(b,format='JPEG',quality=85); return base64.standard_b64encode(b.getvalue()).decode()
def label_image(path):
    return client.messages.parse(model=MODEL,max_tokens=1024,system=SYSTEM,
        output_config={'effort':'medium'},
        messages=[{'role':'user','content':[
            {'type':'image','source':{'type':'base64','media_type':'image/jpeg','data':encode(path)}},
            {'type':'text','text':'Đánh giá độc lập ảnh dừa này cho từng tác vụ.'}]}],
        output_format=ImageLabel).parsed_output
def vote(conf):
    return 1 if conf>=TAU_HIGH else (0 if conf<=TAU_LOW else -1)   # -1 = abstain

## 7. Chạy — cổng trước, LF5 vote sau

In [ ]:
rows=[]
for i,(path,folder) in enumerate(pilot,1):
    q=quality_gate(path)
    row={'image_id':path.stem,'source_folder':folder,'path':str(path.relative_to(ROOT)),**q}
    if q['quality_pass']:
        try: r=label_image(path)
        except Exception as e: print('ERR',path.name,e); continue
        for t in TASKS:
            tk=getattr(r,FIELD[t])
            row[f'lf5_{t}']=vote(tk.confidence)          # phiếu LF5: 1/0/-1(abstain)
            row[f'{t}_conf']=round(tk.confidence,3)
            row[f'{t}_reason']=tk.reason
    else:
        for t in TASKS:
            row[f'lf5_{t}']=-1; row[f'{t}_conf']=np.nan; row[f'{t}_reason']=''  # rớt cổng -> abstain
    rows.append(row)
    if i%10==0 or i==len(pilot): print(f'  {i}/{len(pilot)}')
df=pd.DataFrame(rows); df.to_csv(OUT/'lf5_vision_pilot.csv',index=False)
print('Đã ghi',OUT/'lf5_vision_pilot.csv')
df[['image_id','source_folder','quality_pass']+[f'lf5_{t}' for t in TASKS]].head(10)

## 8. Kiểm tra LF5 vs thư mục native (LF6)

In [ ]:
passed=df[df['quality_pass']]
print(f'Tổng {len(df)} | rớt cổng {int((~df.quality_pass).sum())} | qua cổng {len(passed)}')
print('\nPhiếu LF5 mỗi tác vụ (1=hữu dụng / 0=không / -1=abstain):')
for t in TASKS:
    vc=passed[f'lf5_{t}'].value_counts().to_dict()
    print(f'  {t:22s}: 1={vc.get(1,0)}  0={vc.get(0,0)}  abstain={vc.get(-1,0)}')
# khớp native: trên tác vụ native của thư mục, LF5 vote 1?
print('\nLF5 vote=1 trên tác vụ NATIVE của thư mục (mong đợi cao):')
for folder in passed['source_folder'].unique():
    sub=passed[passed.source_folder==folder]; nat=list(FOLDER_NATIVE[folder])[0]
    pos=int((sub[f'lf5_{nat}']==1).sum())
    print(f'  {folder:18s} -> {nat:22s}: {pos}/{len(sub)}')
print('\nCa cần review (qua cổng, native nhưng LF5 KHÔNG vote 1):')
for _,r in passed.iterrows():
    nat=list(FOLDER_NATIVE[r.source_folder])[0]
    if r[f'lf5_{nat}']!=1:
        print(f"  {r.source_folder:16s} {r.image_id:20s} {nat}: vote={r[f'lf5_{nat}']} conf={r[f'{nat}_conf']}")